# 03. 분류와 군집화

이 노트북에서는 머신러닝의 두 축을 다룹니다.
- **분류(classification)**: 라벨이 있는 데이터로 카테고리를 예측 (지도학습)
- **군집화(clustering)**: 라벨 없이 비슷한 데이터를 묶음 (비지도학습)

scikit-learn 내장 와인 데이터로 두 가지를 모두 실습합니다.

1. 분류 — 로지스틱 회귀와 랜덤 포레스트 비교
2. 분류 평가 — 혼동 행렬
3. 군집화 — K-means + PCA 시각화

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

plt.rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False

## 1. 분류 — 두 모델 비교

데이터를 분할하고 표준화한 뒤, 로지스틱 회귀와 랜덤 포레스트를 각각 학습해 정확도를 비교합니다.
거리·경사 기반 모델(로지스틱)은 변수 스케일에 민감하므로 표준화가 중요합니다.

In [ ]:
data = load_wine()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

logreg = LogisticRegression(max_iter=1000).fit(X_train_s, y_train)
rf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)

print(f'로지스틱 회귀 정확도: {accuracy_score(y_test, logreg.predict(X_test_s))*100:.1f}%')
print(f'랜덤 포레스트 정확도: {accuracy_score(y_test, rf.predict(X_test))*100:.1f}%')

## 2. 분류 평가 — 혼동 행렬

혼동 행렬은 어떤 클래스를 어떤 클래스로 잘못 예측했는지 보여줍니다. 대각선이 정답입니다.

In [ ]:
cm = confusion_matrix(y_test, rf.predict(X_test))
plt.figure(figsize=(5.5, 4.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=data.target_names, yticklabels=data.target_names)
plt.xlabel('예측'); plt.ylabel('실제'); plt.title('혼동 행렬 (랜덤 포레스트)')
plt.tight_layout(); plt.show()

print(classification_report(y_test, rf.predict(X_test), target_names=data.target_names))

## 3. 군집화 — K-means

라벨을 사용하지 않고 데이터를 3개 군집으로 묶습니다. 13차원 데이터를 PCA로 2차원으로 축소해 시각화합니다.
(실제 라벨과 비교하면 군집이 와인 종류를 어느 정도 구분해내는지 볼 수 있습니다.)

In [ ]:
X_scaled = StandardScaler().fit_transform(X)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)

pca = PCA(n_components=2)
X_2d = pca.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(X_2d[:, 0], X_2d[:, 1], c=clusters, cmap='viridis', alpha=0.7)
axes[0].set_title('K-means 군집 결과')
axes[1].scatter(X_2d[:, 0], X_2d[:, 1], c=y, cmap='viridis', alpha=0.7)
axes[1].set_title('실제 라벨')
for ax in axes:
    ax.set_xlabel('주성분 1'); ax.set_ylabel('주성분 2')
plt.tight_layout(); plt.show()

### 정리

- 로지스틱 회귀와 랜덤 포레스트로 분류를 수행하고 혼동 행렬로 평가했습니다.
- K-means로 라벨 없이 군집을 찾고, PCA로 차원을 축소해 결과를 시각화했습니다.
- 분류(지도)와 군집화(비지도)의 차이를 한 데이터로 비교하며 머신러닝의 두 축을 익혔습니다.